# Outcomes: 자신의 작업을 검증하는 agent

agent는 완료된 것처럼 *보이는* 결과물을 만드는 데 능숙합니다. 출처가 있는 조사 보고서를 요청하면 각주가 달린 깔끔한 문서를 받습니다. 자세히 보면 보통 개선의 여지가 있습니다. 어떤 주제는 설명이 부족하고, 인용문은 출처에서 벗어나고, 출처 표시는 원래 공시 대신 보도자료에 의존합니다. 이를 잡으려면 수동 review 루프가 필요합니다. 출력을 읽고, 잘못된 부분을 찾아 다시 prompt를 보냅니다. 그 과정에서 말하는 대부분의 feedback은 agent가 시작하기 전에 적어 둘 수 있는 내용입니다.

Claude Managed Agents의 [Outcomes](https://platform.claude.com/docs/en/managed-agents/define-outcomes)는 세션에 점검만 하는 두 번째 agent를 제공합니다. "완료"의 모습과 검증 방법을 명시하는 **rubric**을 작성하면 플랫폼이 독립적인 context window의 **grader**를 마련합니다. grader는 writer의 reasoning을 볼 수 없고 어떤 지름길을 택했는지도 모릅니다. writer의 각 turn 뒤에 rubric에 따라 artifact를 다시 읽고 통과시키거나 기준별 부족한 부분 목록을 반환합니다. writer가 수정하면 설정한 상한까지 루프가 다시 실행됩니다.

이 guide에서는 그 루프의 전체 실행을 살펴봅니다. writer는 출처의 원문 인용을 포함하여 EV 급속 충전 경제성에 대한 한 페이지 보고서를 작성합니다. grader는 인용된 모든 URL을 독립적으로 fetch하고, 각 페이지에서 인용 문자열을 검색하고, 인용이 실제로 주장을 뒷받침하는지 확인하고, 일곱 항목 checklist로 내용 범위를 평가합니다. 실제 문제(10-K가 필요한 곳에 인용된 보도자료 첨부 문서)를 잡는 모습을 보고 writer가 올바른 문서를 찾는 모습을 보게 됩니다.

## 배울 내용

- grader가 실행할 수 있는 rubric 작성하기
- Outcome이 있는 세션을 시작하고 agent가 이를 향해 작업하게 하기
- 평가 및 수정 루프를 따라가며 grader의 feedback 읽기
- Outcomes가 적합한 상황 판단하기


## 1. environment 설정하기

먼저 SDK를 설치하고 Anthropic client를 설정하겠습니다. `define_outcome` event와 outcome-evaluation span 타입은 Managed Agents beta에 포함됩니다.


In [1]:
%%capture
%pip install -q anthropic python-dotenv

In [2]:
import os
import re
import time

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
client = anthropic.Anthropic()

## 2. writer 생성하고 세션 시작하기

다음으로 writer agent를 생성하고 세션을 엽니다. writer의 system prompt는 최대 여섯 출처를 인용하고 각 출처에 짧은 원문 인용을 붙이도록 요청하여 grader가 구체적으로 확인할 대상을 제공합니다.


In [3]:
env = client.beta.environments.create(
    name="research-brief",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
)

writer = client.beta.agents.create(
    name="Research Analyst",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-verify-with-outcome-grader"},
    model=MODEL,
    system="""You are a research analyst. You write one-page business briefs.

Cite every factual claim with an inline footnote [n]. End the brief with a Sources section in this exact format, one entry per line:

[n] "verbatim quote from the page, 25 words or fewer" - Title - URL

Only cite pages you actually fetched and read. The quote must be copied character-for-character from the page. Cite no more than 6 sources total. Pick the strongest; do not pad. Save the brief to /mnt/session/outputs/brief.md.""",
    tools=[
        {
            "type": "agent_toolset_20260401",
            "configs": [
                {"name": "web_search"},
                {"name": "web_fetch"},
                {"name": "read"},
                {"name": "write"},
            ],
        }
    ],
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": writer.id, "version": writer.version},
    environment_id=env.id,
    title="Brief: EV fast-charging unit economics",
    betas=BETAS,
)
print(f"Session {session.id}")

Session sesn_011CakRYQjc4NMXdRy4qKZy5


## 3. grader가 실행할 수 있는 rubric 작성하기

`define_outcome` event는 세션에 두 가지를 제공합니다. writer가 읽는 결과물의 **description**과 grader가 읽는 **rubric**입니다. writer의 각 turn 뒤에 플랫폼은 writer와 같은 model 및 tool을 가진 새 grader를 시작하고 rubric을 제공하여 artifact를 검사하게 합니다. grader는 기준별 판정을 반환하며, 하나라도 실패하면 설명이 다음 수정을 위해 바로 writer에게 돌아갑니다.

rubric은 grader에 영향을 줄 수 있는 유일한 수단이며 문구가 grader의 실제 점검 여부를 결정합니다. 기본적인 실패 양상은 모든 것을 승인하는 grader입니다. *"보고서가 demand charge를 다루는지 확인하라"*는 rubric에서는 grader가 보고서를 훑고 demand charge 문단을 본 뒤 있다고 확인하는 문장을 쓸 수 있습니다. 단 하나의 출처를 열지 않고도 모두 할 수 있습니다. 이 기준으로는 대부분의 초안이 통과하며 루프는 전혀 실행되지 않습니다. *"보고서를 열고 demand charge 절을 찾아 $/kW 수치나 운영 비용의 %를 명시하는지 확인하라"*는 rubric은 grader가 근거를 만들게 합니다. 그런 grader가 10-K인 척하는 보도자료를 잡습니다.

모든 rubric에서 해 둘 만한 사항은 다음과 같습니다.

| 원칙 | 실제 적용 |
|---|---|
| **각 기준을 확인 가능하게 만들기** | 작업은 "특정 운영자의 경제성"이라고 합니다. rubric은 구체화합니다. `sec.gov`의 공시를 출처로 인용한 10-K 또는 10-Q의 GAAP 순손실입니다. rubric은 항상 작업보다 구체적이어야 합니다. |
| **grader가 근거를 갖춰야 `satisfied`를 주게 하기** | 통과하기 전에 구체적인 근거(fetch한 페이지, 추적한 공식, `file:line` 참조)를 요구하세요. 너무 엄격한 grader는 루프 한 번을 더 발생시킵니다. 너무 관대한 grader는 나쁜 version을 남긴 채 루프를 끝냅니다. |
| **단계 대신 목표 설명하기** | 특정 명령을 지시하는 rubric은 명령을 사용할 수 없을 때 조용히 실패하며 중요한 점검이 실행되지 않습니다. 무엇이 증거인지 정의하세요. grader는 writer의 전체 toolset으로 이를 찾을 수 있습니다. |
| **writer의 지름길 예상하기** | "mirror, 재게시물, 검색 snippet으로 확인하지 마라." 이 문장이 없으면 죽은 출처가 scraper 페이지로 대체되고 grader가 통과시킵니다. |
| **feedback 형식 의무화하기** | grader 설명은 writer의 유일한 신호입니다. 한 줄 평가표를 요구한 뒤, 실패마다 무엇이 잘못되었고 무엇을 해야 하는지 bullet 하나를 요구하세요. |
| **무시할 내용 알려 주기** | 문제로 삼지 않을 목록이 없으면 grader는 사소한 스타일, 기존 issue, scope creep에 매달립니다. 범위 밖의 내용을 명시하고 각 발견을 제기하기 전에 스스로 점검하게 하세요. |

> **아직 rubric이 없나요?** Claude에 잘 만든 것으로 확인된 artifact 예제를 주고 무엇이 좋은지 분석하게 한 다음, 그 분석을 기준으로 바꾸게 하세요. 이 중간 과정은 빈 페이지에서 기준을 쓰는 것보다 안정적으로 낫습니다.

보고서의 작업과 rubric은 아래와 같습니다. 나란히 읽으며 rubric이 얼마나 더 구체적인지 보세요. 작업에 없던 세부 사항을 추가한 모든 곳은 grader가 실제로 실행할 수 있는 점검입니다.


In [4]:
TASK = """Write a brief on the unit economics of public DC fast charging in the United States.
The brief should cover:
  1. Capex range
  2. Demand charges
  3. Utilization breakeven
  4. Subsidy programs
  5. Named-operator economics
  6. A contrarian or skeptical source
  7. Hardware vs install cost split
"""


RUBRIC = """
You are reviewing a research brief at /mnt/session/outputs/brief.md against a coverage checklist and verifying its citations. The writer was told the seven topics to cover; this rubric defines what counts as sufficient coverage for each topic, and how to verify citations.

COVERAGE CHECKLIST. Each item has a specific area:
  1. Capex range: a dollar range for installed cost per DC fast-charging stall or station.
  2. Demand charges: quantified impact on opex (a $/kW figure or a % of operating cost).
  3. Utilization breakeven: a breakeven or target utilization threshold (% or kWh/day).
  4. Subsidy programs: NEVI or another public funding program, named.
  5. Named operator: the GAAP net income or net loss from a specific public charging operator's most recent 10-K or 10-Q, and the citation for it must be the SEC filing itself (sec.gov), not a press release, earnings-call recap, or news article.
  6. Contrarian source: at least one cited source whose thesis is that the economics are unfavorable or structurally challenged.
  7. Cost split: a hardware vs soft-cost (install, permitting, grid) breakdown or ratio.

CITATION CHECK. For every [n] entry in the Sources section:
  a. LIVE: Fetch the URL with web_fetch. Mark LIVE only if web_fetch returns the readable page directly. Mark DEAD if 404, parked, login-walled, paywalled, returns a bot-block/403, or renders only via JavaScript. Do NOT corroborate via mirrors, reposts, or search snippets; the cited URL itself must fetch.
  b. VERBATIM: Search the fetched page for the quoted string. Mark QUOTE_MATCH if the exact string appears (treat curly vs straight quotes as equivalent); NOT_FOUND otherwise.
  c. SUPPORTS CLAIM: Mark SUPPORTS_CLAIM if the quoted passage actually backs the claim it's cited on in the brief; UNSUPPORTED if it's tangential, contradicts the claim, or is just a general statement of fact.

OUTPUT FORMAT:

Line 1: Coverage N/7. Citations M/K verified.

Then, for each failed item in the coverage checklist, create a new bullet, name the item and say what specific bar it failed in one sentence max per bullet. For example: "Item 3 Utilization breakeven - MISSING. <what's missing>".

Then, for each failed citation, create a new bullet with the format: "[n] domain - REASON. <what's wrong and what to do>". One sentence max per bullet. For example: "[3] evgo.com - DEAD. The URL returns a 403 error and appears to be behind a bot block. No mirrors or reposts; the cited URL itself must fetch."
"""

client.beta.sessions.events.send(
    session.id,
    betas=BETAS,
    events=[
        {
            "type": "user.define_outcome",
            "description": TASK,
            "rubric": {"type": "text", "content": RUBRIC},
            "max_iterations": 5,
        },
    ],
)

BetaManagedAgentsSendSessionEvents(data=[BetaManagedAgentsUserDefineOutcomeEvent(id='sevt_011CakRYTKNkXJN7eh4kRs1r', description='Write a brief on the unit economics of public DC fast charging in the United States.\nThe brief should cover:\n  1. Capex range\n  2. Demand charges\n  3. Utilization breakeven\n  4. Subsidy programs\n  5. Named-operator economics\n  6. A contrarian or skeptical source\n  7. Hardware vs install cost split\n', max_iterations=5, outcome_id='outc_011CakRYTKNkJ83CBexcz13e', processed_at=datetime.datetime(2026, 5, 6, 1, 10, 30, 512206, tzinfo=TzInfo(0)), rubric=BetaManagedAgentsTextRubric(content='\nYou are reviewing a research brief at /mnt/session/outputs/brief.md against a coverage checklist and verifying its citations. The writer was told the seven topics to cover; this rubric defines what counts as sufficient coverage for each topic, and how to verify citations.\n\nCOVERAGE CHECKLIST. Each item has a specific area:\n  1. Capex range: a dollar range for insta

위 호출에 대한 참고 사항입니다.

- **description과 rubric의 역할은 다릅니다.** description은 writer에게 무엇을 만들지 알려 줍니다. rubric은 grader에게 어떻게 확인할지 알려 줍니다. artifact 위치와 형식은 일치해야 합니다. 서로 모순되면(예: description은 inline 출력을 요청하고 rubric은 `/mnt/session/outputs/` 파일을 평가함) 루프는 헛돌지 않고 `failed`를 반환합니다.
- **`max_iterations`의 기본값은 3(최대 20)입니다.** 이 rubric이 엄격하고 writer에게 여유가 필요하므로 5로 설정합니다. 매 실행이 상한에 도달하고 grader가 매번 같은 종류의 문제를 찾는다면 writer가 feedback을 실행할 수 없는 것이며 수렴하지 않는 반복에 비용을 지불하고 있습니다.
- **inline rubric과 파일 rubric.** notebook에는 inline text가 적합합니다. production에서는 Files API로 rubric을 한 번 업로드하고 `rubric: {"type": "file", "file_id": ...}`를 전달하여 세션 사이에 재사용하고 코드처럼 검토할 수 있게 합니다.

**rubric을 system prompt에 넣기만 하면 안 될까요?** 가능합니다. writer가 목표를 더 잘 맞추는 데 도움이 됩니다. 하지만 기준을 아는 writer도 여전히 자신의 작업을 평가합니다. 통과했다고 믿을 때마다 통과했다고 말하며, 이미 인용한 URL을 다시 fetch하거나 기억하는 인용문이 페이지의 인용문과 조금 다르다는 것을 알아차리지 않습니다. grader는 그 점검을 해야만 합니다. rubric과 artifact만 있는 새 context window로 시작하며, 플랫폼은 모든 기준의 판정이 나올 때까지 루프 진행을 허용하지 않습니다. 아무리 잘 작성해도 단일 prompt로 이런 분리를 얻을 수는 없습니다.


## 4. review 루프 관찰하기

event를 streaming하고 각 단계를 진행되는 대로 표시하겠습니다. writer가 초안을 완료하면 banner를 출력하고 각 평가 뒤 grader의 feedback을 보여 줍니다.


아래 루프를 위한 표시 helper 두 개입니다. `banner`는 이름이 있는 구분선을 그리며 `render_feedback`은 서버가 각 grader 설명에 붙이는 상용 문구를 제거합니다.


In [5]:
HR = "━" * 46


def banner(label, tag=""):
    print(f"\n{HR}\n{label}  {tag}".rstrip())


def render_feedback(fb: str):
    # 서버가 붙인 기준별 앞뒤 문구를 제거합니다.
    s = re.sub(
        r"^An independent grader found.*?:\n\n- .*?\((?:partially |not )?met\): ",
        "",
        fb,
        count=1,
        flags=re.S,
    )
    s = re.sub(r"\n\nPlease revise your work.*$", "", s, flags=re.S)
    print(s)

In [6]:
TERMINAL = {"satisfied", "max_iterations_reached", "failed", "interrupted"}
t0, res, iters = time.time(), None, 0
n_search, last_len = 0, 0

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "agent.tool_use":
            if ev.name in ("web_search", "web_fetch"):
                n_search += 1
            if ev.name == "write" and ev.input["file_path"].endswith("brief.md"):
                last_len = len(ev.input["content"])
        elif ev.type == "span.outcome_evaluation_start":
            banner("writer · " + ("draft" if iters == 0 else f"revision {iters}"))
            print(f"searched/fetched {n_search}× · wrote brief.md ({last_len:,} chars)")
            n_search = 0
        elif ev.type == "span.outcome_evaluation_end":
            res = ev.result
            banner(
                f"grader · pass {iters}",
                "✓ satisfied" if res == "satisfied" else "⟳ needs_revision",
            )
            render_feedback(ev.explanation)
            iters += 1
            if res in TERMINAL:
                break

m, s = divmod(int(time.time() - t0), 60)
print(f"\ndone: {res} after {iters} pass{'es' if iters != 1 else ''} · {m}m {s:02d}s")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
writer · draft
searched/fetched 18× · wrote brief.md (7,607 chars)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
grader · pass 0  ⟳ needs_revision
The brief covers 5 of 7 required topics adequately. Item 2 (Demand charges) fails the quantification bar: the brief describes demand charges qualitatively as the 'single largest operational wildcard' but never states a $/kW figure (McKinsey's footnote of $20/kW is never quoted in the text) or a % of operating cost. Item 5 (Named operator) fails the citation requirement: EVgo's Q1 2024 net loss of $28.2M is cited to evchargingstations.com [6], a third-party news article, not an SEC filing from sec.gov as the rubric explicitly requires. All 6 citations are LIVE, and all 6 quoted strings match the fetched pages and support the claims they are attached to — but [6] is the wrong source type for the Named Operator criterion.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
writer · revision 1
search

### 방금 일어난 일

루프는 세 차례 평가를 수행했습니다.

첫 초안은 일곱 항목 중 다섯 개를 다뤘습니다. grader는 두 누락 때문에 돌려보냈습니다. demand charge가 kW당 달러 수치 없이 정성적으로 설명되어 있었고, 특정 운영자 절이 SEC 공시 대신 제3자 뉴스 기사를 인용했습니다. writer는 $20/kW 수치를 추가하고 sec.gov에서 EVgo의 FY2024 순손실을 찾았습니다.

두 번째 평가는 일곱 개 중 여섯 개를 통과했습니다. writer가 인용한 sec.gov 문서는 rubric이 요구한 10-K나 10-Q가 아니라 공시된 실적 보도자료인 8-K Exhibit 99.1이었습니다. grader는 URL을 읽고 보도자료 첨부 문서로 판별하여 정확히 그 차이를 이유로 거부했습니다. 세 번째 평가에서 writer는 EDGAR에서 EVgo의 실제 10-K를 찾았고 grader가 통과시켰습니다. 7/7 내용 범위, 6/6 접근 가능한 인용 출처, 일치하며 주장을 뒷받침하는 인용문이었습니다.

두 거부 모두 rubric이 grader가 확인할 수 있는 선을 그었기 때문에 발생했습니다. "$/kW 수치나 운영 비용의 %"와 "보도자료 대신 SEC 공시 자체"는 rubric 작성자가 정한 사항입니다. 작업 자체("demand charge 다루기", "특정 운영자의 경제성")라면 둘 다 통과시켰을 것입니다. 두 번째 거부를 살펴볼 필요가 있습니다. writer는 겉으로는 작업을 만족하는 `sec.gov` URL을 찾았는데도 rubric이 8-K 보도자료 첨부 문서와 10-K를 구분했으므로 grader가 돌려보냈습니다. 이 구분이 없으면 더 나쁜 보고서로 루프가 한 차례 일찍 끝납니다.

위 trace는 원래 cookbook 실행에서 가져온 것이라는 점에 **유의하세요**. 직접 실행하면 결과가 다를 가능성이 높습니다.


## 5. 최종 보고서 읽기

마지막으로 writer가 만든 `brief.md`의 마지막 version을 가져와 무엇을 다루고 무엇을 인용했는지 살펴보겠습니다. 전체 문장을 읽으려면 `content` 자체를 출력하세요.


In [7]:
# event log에서 최종 보고서를 재구성합니다. `write`의 전체 내용에
# 각 `edit`(old_string -> new_string)을 순서대로 적용합니다.
content = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if ev.type != "agent.tool_use" or "brief.md" not in str(ev.input.get("file_path", "")):
        continue
    if ev.name == "write":
        content = ev.input["content"]
    elif ev.name == "edit":
        content = content.replace(ev.input["old_string"], ev.input["new_string"], 1)

# 전체 문장 대신 구조와 출처를 보여 줍니다.
for line in content.splitlines():
    if line.startswith(("#", "[")):
        print(line)

# Unit Economics of Public DC Fast Charging in the United States
## 1. Capex Range
## 2. Demand Charges
## 3. Utilization Breakeven
## 4. Subsidy Programs
## 5. Named-Operator Economics
## 6. Contrarian / Skeptical View
## 7. Hardware vs. Installation Cost Split
## Sources
[1] "A 150 to 350kW DCFC charging unit can cost anywhere from $45,000 to over $100,000, and installation costs can range from $40,000 to over $150,000." - Can public EV fast-charging stations be profitable in the United States? - https://www.mckinsey.com/features/mckinsey-center-for-future-mobility/our-insights/can-public-ev-fast-charging-stations-be-profitable-in-the-united-states
[2] "Waiving demand charges for fast chargers will shift costs to all ratepayers, including non-EV drivers." - Fast charging, high costs: Eliminating demand charges won't solve the problem - https://www.utilitydive.com/news/eliminating-demand-charges-wont-solve-EV-station-problems/689395/
[3] "the average total NEVI project cost is $915,42

## 배운 내용

- **무엇이 좋은 결과인지 적을 수 있을 때 Outcomes가 적합합니다.** 세심한 주의와 빠짐없는 내용 범위가 필요한 작업이 가장 명확하게 맞습니다. rubric이 기준을 구체화하면 주관적인 품질도 가능합니다.
- **rubric은 grader가 근거를 생성하게 해야 합니다.** 그렇지 않으면 보여 주는 모든 것을 승인합니다.
- **grader는 독립적이며 stateless입니다.** 자체 context window에서 실행되므로 writer가 설득할 수 없고 매 반복마다 새 grader가 전체 artifact를 다시 점검합니다.
- **한 번에 outcome 하나지만 이어서 연결할 수 있습니다.** 루프가 종료하면 세션은 다시 대화형이 되고 다음 `user.define_outcome`이 새 루프를 시작합니다.

자세한 내용은 [Outcomes 문서](https://platform.claude.com/docs/en/managed-agents/define-outcomes)를 참고하세요.

